# Exploratory Analysis of Drug–Target Interaction Datasets

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from rdkit import Chem
from rdkit.Chem import Descriptors

from scipy.stats import pearsonr

In [ ]:
ROOT_DIR = Path("../").resolve()

DATA_DIR = ROOT_DIR / "datasets"

RAW_DIR = DATA_DIR / "raw"

PROCESSED_DIR = DATA_DIR / "processed"

print(ROOT_DIR)

## Load Dataset

In [ ]:
bindingdb = pd.read_csv(
    RAW_DIR / "bindingdb.csv"
)

bindingdb.head()

In [ ]:
print("Rows:", len(bindingdb))
print("Columns:", len(bindingdb.columns))

bindingdb.info()

## Missing Value Analysis

In [ ]:
missing = bindingdb.isnull().sum()

missing = missing[
    missing > 0
]

missing.sort_values(
    ascending=False
)

In [ ]:
plt.figure(figsize=(10,5))

missing.sort_values(
    ascending=False
).plot.bar()

plt.title("Missing Values")

plt.show()

## Label Distribution (BindingDB)

In [ ]:
bindingdb["label"].value_counts()

In [ ]:
plt.figure(figsize=(6,4))

bindingdb["label"].value_counts().plot.bar()

plt.title("BindingDB Label Distribution")

plt.show()

## DAVIS Distribution

In [ ]:
davis = pd.read_csv(
    RAW_DIR / "davis.csv"
)

davis["affinity"].describe()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    davis["affinity"],
    bins=50
)

plt.title("DAVIS Affinity Distribution")

plt.show()

## KIBA Distribution

In [ ]:
kiba = pd.read_csv(
    RAW_DIR / "kiba.csv"
)

kiba["affinity"].describe()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    kiba["affinity"],
    bins=50
)

plt.title("KIBA Score Distribution")

plt.show()

## Molecular Weight Analysis

In [ ]:
def molecular_weight(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return np.nan

    return Descriptors.MolWt(mol)

bindingdb["mol_weight"] = (
    bindingdb["smiles"]
    .apply(molecular_weight)
)

## Protein Length Analysis

In [ ]:
bindingdb["protein_length"] = (
    bindingdb["sequence"]
    .str.len()
)

bindingdb["protein_length"].describe()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    bindingdb["protein_length"],
    bins=50
)

plt.title("Protein Length Distribution")

plt.show()

## Correlation Matrix

In [ ]:
numeric_columns = bindingdb.select_dtypes(
    include=np.number
)

corr = numeric_columns.corr()

corr

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    cmap="coolwarm"
)

plt.title("Correlation Matrix")

plt.show()

In [ ]:
bindingdb["smiles"].value_counts().head(20)

In [ ]:
bindingdb["sequence"].value_counts().head(20)

In [ ]:
summary = pd.DataFrame({

    "Dataset": [
        "BindingDB",
        "DAVIS",
        "KIBA"
    ],

    "Samples": [
        len(bindingdb),
        len(davis),
        len(kiba)
    ]
})

summary

In [ ]:
summary.to_csv(
    "dataset_summary.csv",
    index=False
)

## PCA Visualization

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

features = numeric_columns.fillna(0)

scaled = StandardScaler().fit_transform(
    features
)

pca = PCA(
    n_components=2
)

coords = pca.fit_transform(
    scaled
)

plt.figure(figsize=(8,6))

plt.scatter(
    coords[:,0],
    coords[:,1],
    s=5
)

plt.title("PCA Projection")

plt.show()